# 📓 Exercise 03 — Semantic Search & Embeddings

**Series:** RAG Foundations | **Difficulty:** ⭐⭐ Beginner-Intermediate  
**Time:** ~40 minutes

---
## 🎯 Learning Objectives
1. Understand what embeddings are and how they encode meaning
2. Use `sentence-transformers` to create embeddings
3. Compare all three semantic distance metrics: cosine, dot product, Euclidean
4. Implement brute-force k-NN retrieval
5. Build and query a FAISS HNSW index (production ANN)
6. Visualise embedding spaces with PCA/t-SNE

---
## 📖 Concept: From Words to Vectors

### The Problem with Keyword Search
> Query: `"affordable lunar expedition"`  
> Document: `"cheapest moon mission"`

TF-IDF and BM25 score this **0** — zero shared words!  
But the meaning is identical.

### The Solution: Embeddings
An **embedding** is a dense numeric vector (e.g., 384 numbers) that represents the *meaning* of text.

- Similar meanings → vectors close together in space
- Different meanings → vectors far apart

```
"affordable lunar expedition" → [0.12, -0.45, 0.78, ...]  (384 numbers)
"cheapest moon mission"       → [0.13, -0.43, 0.80, ...]  (very similar!)
"ice cream flavours"          → [-0.91, 0.23, -0.15, ...] (very different)
```

---

In [ ]:
!pip install sentence-transformers faiss-cpu numpy matplotlib scikit-learn --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from numpy.linalg import norm
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import faiss
import time

# Load embedding model (downloads ~90MB on first run)
print("Loading embedding model (this takes ~30 seconds on first run)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Model loaded! Embedding dimension: {model.get_sentence_embedding_dimension()}")

---
## 🔬 Part 1: Understanding Embeddings

In [ ]:
# ── DEMO: What does an embedding look like? ──

text = "ISRO launched the cheapest Moon mission"
embedding = model.encode(text)

print(f"Text: {text!r}")
print(f"\nEmbedding shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10].round(4)}")
print(f"Min value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")
print(f"L2 norm (length): {norm(embedding):.4f}")

# Visualise first 50 dimensions
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(50), embedding[:50], color=['#4A90D9' if v > 0 else '#E74C3C' for v in embedding[:50]], alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Dimension index (first 50 of 384)')
ax.set_ylabel('Value')
ax.set_title(f'Embedding Vector: {text!r}\n(384-dimensional dense vector — first 50 shown)')
plt.tight_layout()
plt.savefig('embedding_vector.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: embedding_vector.png")

In [ ]:
# ── DEMO: Similar phrases → similar embeddings ──

phrase_pairs = [
    # Very similar
    ("cheapest moon mission",       "affordable lunar expedition"),
    ("happy joyful people",         "cheerful delighted individuals"),
    ("artificial intelligence AI",  "machine learning deep learning"),
    # Different meaning, same words sometimes
    ("bank of a river",             "financial bank account"),
    # Completely unrelated
    ("space shuttle launch",        "chocolate cake recipe"),
    ("quantum physics theory",      "children playing football"),
]

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(f"{'Phrase 1':<35} {'Phrase 2':<35} {'Cosine Sim':>12}")
print('-' * 85)
for p1, p2 in phrase_pairs:
    e1 = model.encode(p1)
    e2 = model.encode(p2)
    sim = cosine_similarity(e1, e2)
    bar = '█' * int(sim * 20) if sim > 0 else ''
    print(f"  {p1[:33]:<33} {p2[:33]:<33} {sim:>8.4f}  {bar}")

print("\n📝 Higher cosine similarity = more similar meaning")
print("   'bank' example shows embeddings capture CONTEXT, not just words")

### ✏️ Exercise 1.1 — Semantic pairing challenge

Which pairs will have the highest similarity? Predict before running:

In [ ]:
# ✏️ Exercise 1.1 — Before running, predict the ORDER from highest to lowest similarity
# Your prediction: ___________

pairs_to_rank = [
    ("dog", "puppy"),
    ("dog", "cat"),
    ("dog", "animal"),
    ("dog", "car"),
    ("dog", "bark"),
    ("dog", "wolf"),
]

sims = []
for p1, p2 in pairs_to_rank:
    sim = cosine_similarity(model.encode(p1), model.encode(p2))
    sims.append((sim, p1, p2))
sims.sort(reverse=True)

print("Ranked from most to least similar:")
for i, (sim, p1, p2) in enumerate(sims, 1):
    bar = '█' * int(sim * 25)
    print(f"  {i}. '{p1}' ↔ '{p2}': {sim:.4f}  {bar}")

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{p1} ↔ {p2}" for _, p1, p2 in sims]
values = [s for s, _, _ in sims]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sims)))[::-1]
ax.barh(labels[::-1], values[::-1], color=colors[::-1], alpha=0.85)
ax.set_xlabel('Cosine Similarity')
ax.set_title('Exercise 1.1 — Semantic Similarity Rankings')
ax.set_xlim(0, 1)
for i, v in enumerate(values[::-1]):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 🔬 Part 2: Scoring Functions — Cosine, Dot Product, Euclidean

In [ ]:
# ── All three similarity/distance metrics explained ──

space_docs = [
    "NASA Apollo 11 first humans on the Moon 1969.",
    "ISRO Chandrayaan cheapest successful Moon mission water discovery.",
    "ESA Rosetta probe comet mission landing.",
    "Hubble Space Telescope deep space images galaxies.",
    "Mars rovers Curiosity Perseverance exploring Martian surface.",
]

query = "Which mission discovered water on the Moon?"

q_emb  = model.encode(query)
d_embs = model.encode(space_docs)

# Metric 1: Cosine Similarity
def cosine_sim(a, b):
    """Measures angle between vectors. Range: [-1, 1]. Higher = more similar."""
    return np.dot(a, b) / (norm(a) * norm(b))

# Metric 2: Dot Product
def dot_product(a, b):
    """Measures projection. Depends on vector magnitude (length)."""
    return np.dot(a, b)

# Metric 3: Euclidean Distance
def euclidean(a, b):
    """Straight-line distance. LOWER = more similar (opposite of cosine)."""
    return norm(a - b)

print(f"Query: {query!r}\n")
print(f"{'Doc':<5} {'Cosine↑':>10} {'DotProd↑':>10} {'Euclidean↓':>12}  Text")
print('-' * 85)

rows = []
for i, (doc, d_emb) in enumerate(zip(space_docs, d_embs)):
    cs = cosine_sim(q_emb, d_emb)
    dp = dot_product(q_emb, d_emb)
    eu = euclidean(q_emb, d_emb)
    rows.append((i, cs, dp, eu))
    print(f"  D{i+1}   {cs:>10.4f} {dp:>10.4f} {eu:>12.4f}  {doc[:50]}")

# Who wins by each metric?
best_cos = max(rows, key=lambda x: x[1])
best_dot = max(rows, key=lambda x: x[2])
best_euc = min(rows, key=lambda x: x[3])

print(f"\n🏆 Best by Cosine:    Doc {best_cos[0]+1}")
print(f"🏆 Best by Dot Product: Doc {best_dot[0]+1}")
print(f"🏆 Best by Euclidean:   Doc {best_euc[0]+1}")
print("\n📝 For normalised embeddings (||v||=1), cosine and dot product are equivalent.")

In [ ]:
# ── Visualise: all 3 metrics side by side ──
import pandas as pd

cos_scores = np.array([cosine_sim(q_emb, d) for d in d_embs])
euc_scores = np.array([euclidean(q_emb, d)  for d in d_embs])
# Convert Euclidean to similarity (smaller = better, so invert)
euc_sim = 1 / (1 + euc_scores)

x = np.arange(len(space_docs))
labels = [f"D{i+1}" for i in range(len(space_docs))]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, scores, title, color in [
    (axes[0], cos_scores, 'Cosine Similarity (↑ better)',  '#4A90D9'),
    (axes[1], cos_scores, 'Dot Product (↑ better)',        '#27AE60'),  # same for normalised
    (axes[2], euc_sim,    '1/(1+Euclidean) (↑ better)',   '#E67E22'),
]:
    bars = ax.bar(x, scores, color=color, alpha=0.85)
    best_idx = np.argmax(scores)
    bars[best_idx].set_edgecolor('#E74C3C')
    bars[best_idx].set_linewidth(3)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title(title, fontsize=9)
    for i, s in enumerate(scores):
        ax.text(i, s + 0.005, f'{s:.3f}', ha='center', fontsize=8)

plt.suptitle(f'Scoring Functions Comparison\nQuery: {query}', fontsize=10)
plt.tight_layout()
plt.savefig('scoring_functions.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: scoring_functions.png | Red border = best match")

---
## 🔬 Part 3: k-NN Brute Force Retrieval

In [ ]:
# ── k-NN: Brute force semantic search ──

knowledge_base = [
    "NASA launched Apollo 11 and sent the first humans to the Moon in 1969.",
    "ISRO's Chandrayaan proved the Moon has water ice and was the cheapest lunar mission.",
    "SpaceX developed reusable Falcon 9 rockets to dramatically reduce launch costs.",
    "The Hubble Space Telescope captures stunning deep-space images of distant galaxies.",
    "Mars rovers like Curiosity and Perseverance have explored Mars for over a decade.",
    "ESA's Rosetta probe successfully landed on a comet for the first time in history.",
    "The James Webb Space Telescope can observe galaxies formed just after the Big Bang.",
    "SpaceX Starship aims to carry the first humans to Mars within this decade.",
]

class KNNRetriever:
    """
    Brute-force k-NN semantic retriever.
    
    How it works:
    1. Pre-compute embeddings for all documents (offline step)
    2. At query time: embed the query, compute similarity to ALL docs, return top-k
    
    Complexity: O(N×D) per query  (N=docs, D=embedding dim)
    Good for: < 10,000 documents
    Bad for: millions of documents (use HNSW/ANN instead)
    """
    
    def __init__(self, documents, model):
        self.documents = documents
        self.model     = model
        print("Computing document embeddings...")
        self.doc_embeddings = model.encode(documents)
        print(f"✅ Indexed {len(documents)} documents (shape: {self.doc_embeddings.shape})")
    
    def search(self, query, k=3):
        """
        Find top-k most semantically similar documents.
        
        Parameters:
            query : str — the search query
            k     : int — number of results to return
        Returns:
            list of (score, document) tuples
        """
        query_emb = self.model.encode(query)
        sims = [cosine_sim(query_emb, d) for d in self.doc_embeddings]
        top_idx = np.argsort(-np.array(sims))[:k]
        return [(sims[i], self.documents[i]) for i in top_idx]


knn = KNNRetriever(knowledge_base, model)

# Semantic queries that would FAIL with keyword search
semantic_queries = [
    "affordable expedition to the lunar surface",          # → should find ISRO/Chandrayaan
    "first person who walked on the moon",                 # → should find Apollo 11
    "spacecraft studying a moving icy space rock",        # → should find ESA/Rosetta/comet
    "next generation infrared space observatory",         # → should find James Webb
]

print("\n" + "="*60)
print("k-NN SEMANTIC RETRIEVAL DEMO")
print("(These queries have ZERO keyword overlap with the answers!)")
print("="*60)

for query in semantic_queries:
    print(f"\n🔍 Query: {query!r}")
    results = knn.search(query, k=2)
    for i, (score, doc) in enumerate(results, 1):
        print(f"  {i}. (cos={score:.4f}) {doc}")

---
## 🔬 Part 4: ANN with FAISS HNSW — Production Scale

In [ ]:
# ── FAISS HNSW: Hierarchical Navigable Small World ──

# What is HNSW?
# - Organises vectors as nodes in a graph
# - Multiple LAYERS: top layer = sparse (few long-range connections)
#                    bottom layer = dense (many short-range connections)
# - Search: start coarse at top → refine → zoom in at bottom
# - Result: near-exact nearest neighbours, but MUCH faster than brute force

# Build FAISS HNSW index
doc_embs = model.encode(knowledge_base).astype('float32')
dim = doc_embs.shape[1]  # 384

# M = number of connections per node (higher = more accurate but more memory)
# efSearch = search quality at query time (higher = more accurate but slower)
M = 16
index_hnsw = faiss.IndexHNSWFlat(dim, M)
index_hnsw.hnsw.efSearch = 50  # search quality

# Add documents to index
index_hnsw.add(doc_embs)
print(f"✅ FAISS HNSW index built: {index_hnsw.ntotal} vectors, dim={dim}, M={M}")

def hnsw_search(query_text, index, documents, model, k=3):
    """
    Search using FAISS HNSW approximate nearest neighbours.
    
    Note: Returns L2 (Euclidean) distances by default.
    Smaller distance = more similar.
    """
    q_emb = model.encode([query_text]).astype('float32')
    distances, indices = index.search(q_emb, k)
    return [(distances[0][i], documents[indices[0][i]]) for i in range(k)]

# Test the HNSW index
test_q = "cheapest way to reach the moon"
print(f"\nHNSW Search: {test_q!r}")
for i, (dist, doc) in enumerate(hnsw_search(test_q, index_hnsw, knowledge_base, model), 1):
    print(f"  {i}. (L2 dist={dist:.4f}) {doc}")

In [ ]:
# ── Scalability benchmark: brute-force k-NN vs HNSW ──

# Scale up to simulate larger collections
scale_tests = [100, 500, 1000, 2000]
base_embs = model.encode(knowledge_base).astype('float32')

flat_times = []
hnsw_times = []

for n_docs in scale_tests:
    # Tile embeddings to reach desired size
    repeats = math.ceil(n_docs / len(knowledge_base))
    big_embs = np.tile(base_embs, (repeats, 1))[:n_docs].astype('float32')
    
    # Flat (exact) index
    flat_idx = faiss.IndexFlatL2(dim)
    flat_idx.add(big_embs)
    
    # HNSW index
    hnsw_idx = faiss.IndexHNSWFlat(dim, 16)
    hnsw_idx.add(big_embs)
    
    q_emb = model.encode(["moon mission"]).astype('float32')
    
    # Time flat
    t0 = time.time()
    for _ in range(50): flat_idx.search(q_emb, 5)
    flat_times.append((time.time() - t0) / 50 * 1000)
    
    # Time HNSW
    t0 = time.time()
    for _ in range(50): hnsw_idx.search(q_emb, 5)
    hnsw_times.append((time.time() - t0) / 50 * 1000)

import math as _math
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(scale_tests, flat_times, 'o-', color='#E74C3C', linewidth=2, label='Flat (exact k-NN)', markersize=8)
ax.plot(scale_tests, hnsw_times, 's-', color='#27AE60', linewidth=2, label='HNSW (ANN)',        markersize=8)

for x, ft, ht in zip(scale_tests, flat_times, hnsw_times):
    ax.annotate(f'{ft:.2f}ms', (x, ft), textcoords='offset points', xytext=(5, 5), fontsize=8, color='#E74C3C')
    ax.annotate(f'{ht:.2f}ms', (x, ht), textcoords='offset points', xytext=(5,-12), fontsize=8, color='#27AE60')

ax.set_xlabel('Number of Documents in Index')
ax.set_ylabel('Average Query Time (ms)')
ax.set_title('Exact k-NN vs HNSW: Query Latency Scaling')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('knn_vs_hnsw.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: knn_vs_hnsw.png")

---
## 🔬 Part 5: Visualising Embedding Space

In [ ]:
# ── PCA: Project 384-dim embeddings to 2D for visualisation ──

topic_groups = {
    'Space Missions':  ['moon landing Apollo', 'Mars rover mission', 'comet probe landing', 'lunar orbit mission'],
    'Telescopes':      ['Hubble space telescope', 'James Webb infrared telescope', 'radio telescope astronomy', 'optical observatory'],
    'Rockets':         ['reusable rocket launch', 'SpaceX Falcon 9', 'rocket fuel propulsion', 'launch vehicle payload'],
    'Food & Cooking':  ['chocolate cake baking', 'pasta carbonara recipe', 'grilled chicken dinner', 'vegetable soup cooking'],
    'Sports':          ['football match goal', 'tennis grand slam', 'basketball dunk score', 'swimming race competition'],
}

all_texts  = [text for texts in topic_groups.values() for text in texts]
all_labels = [label for label, texts in topic_groups.items() for _ in texts]

embeddings = model.encode(all_texts)
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embeddings)

colors = {'Space Missions': '#4A90D9', 'Telescopes': '#27AE60',
          'Rockets': '#E67E22', 'Food & Cooking': '#E74C3C', 'Sports': '#8E44AD'}

fig, ax = plt.subplots(figsize=(10, 7))
for label, group_texts in topic_groups.items():
    idxs = [i for i, lbl in enumerate(all_labels) if lbl == label]
    ax.scatter(coords_2d[idxs, 0], coords_2d[idxs, 1],
               c=colors[label], s=120, label=label, alpha=0.85, zorder=3)
    for idx in idxs:
        ax.annotate(all_texts[idx][:18] + '..', coords_2d[idx],
                    textcoords='offset points', xytext=(5, 3), fontsize=7, alpha=0.7)

ax.set_title('2D PCA of Sentence Embeddings\nSimilar meanings cluster together!', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(loc='best')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('embedding_space_pca.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: embedding_space_pca.png")
print("\n📝 Notice how space topics cluster together,")
print("   and food/sports are clearly separated from science topics!")

### ✏️ Exercise 5.1 — Custom domain embedding space

In [ ]:
# ✏️ YOUR TURN — Create your own topic clusters
# Add 3-5 topics with 4 phrases each and visualise the embedding space

my_topic_groups = {
    # TODO: Replace with your own topics and phrases!
    'Technology': ['machine learning model training', 'neural network deep learning', 'GPU compute training', 'transformer architecture'],
    'Medicine':   ['cancer treatment therapy', 'vaccine immunology response', 'drug clinical trial', 'patient surgery recovery'],
    'Finance':    ['stock market investment', 'interest rate inflation', 'cryptocurrency bitcoin', 'portfolio diversification'],
}

my_texts  = [t for texts in my_topic_groups.values() for t in texts]
my_labels = [l for l, texts in my_topic_groups.items() for _ in texts]

my_embs = model.encode(my_texts)
pca2 = PCA(n_components=2, random_state=42)
my_2d = pca2.fit_transform(my_embs)

fig, ax = plt.subplots(figsize=(9, 6))
my_colors = ['#4A90D9', '#E74C3C', '#27AE60', '#E67E22', '#8E44AD']
for i, (label, group_texts) in enumerate(my_topic_groups.items()):
    idxs = [j for j, lbl in enumerate(my_labels) if lbl == label]
    ax.scatter(my_2d[idxs, 0], my_2d[idxs, 1], c=my_colors[i], s=120, label=label, alpha=0.85, zorder=3)
    for idx in idxs:
        ax.annotate(my_texts[idx][:20], my_2d[idx], textcoords='offset points', xytext=(4, 3), fontsize=8)

ax.set_title('Your Custom Embedding Space')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---
## 📋 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Embedding** | Dense numeric vector (e.g., 384 dims) representing text meaning |
| **Cosine Similarity** | Angle between vectors; best for comparing meaning regardless of length |
| **Dot Product** | Faster; equals cosine when vectors are normalised |
| **Euclidean** | Straight-line distance; smaller = more similar |
| **k-NN** | Exact search; O(N×D) — too slow for millions of docs |
| **HNSW (ANN)** | Multi-layer graph; near-exact results in milliseconds |
| **Semantic gap** | Embeddings bridge words with different forms but same meaning |

In [ ]:
print("="*60)
print("EXERCISE 03 — SEMANTIC SEARCH: REFERENCE")
print("="*60)
print()
print("SCORING FUNCTIONS:")
print("  Cosine Sim  = dot(a,b) / (||a|| × ||b||)  → [-1, 1] ↑")
print("  Dot Product = sum(a_i × b_i)              → real    ↑")
print("  Euclidean   = sqrt(sum((a-b)^2))           → [0,∞)  ↓")
print()
print("ANN METHODS:")
print("  HNSW  → graph-based, most popular, used in Pinecone/Weaviate")
print("  IVF   → cluster-based, good for very large datasets")
print("  PQ    → product quantisation, reduces memory footprint")
print()
print("WHEN SEMANTIC BEATS KEYWORD:")
print("  Synonyms: 'car' vs 'automobile'")
print("  Paraphrases: 'cheapest' vs 'most affordable'")
print("  Implicit meaning: 'astronaut' relates to 'Moon mission'")
print("="*60)